# SFT, DPO, RLHF — Fine-Tuning LLMs for Marketing

Companion notebook for the blog post: [SFT, DPO, RLHF — Picking the Right Fine-Tuning Strategy for Your LLM](https://cmenguy.github.io/llm/fine-tuning/2026/03/02/fine-tuning-llms-sft-dpo-rlhf/)

This notebook walks through all three fine-tuning approaches with synthetic marketing data. Each section is self-contained and can be run independently.

## Setup

Install all required dependencies. If running on Colab, the GPU runtime is recommended.

In [ ]:
!pip install -q torch transformers datasets peft trl accelerate bitsandbytes

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig
from trl import (
    SFTConfig, SFTTrainer,
    DPOConfig, DPOTrainer,
    RewardConfig, RewardTrainer,
    RLOOConfig, RLOOTrainer,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Part 1: Supervised Fine-Tuning (SFT)

SFT minimizes the standard cross-entropy loss over curated examples. It learns to imitate your best marketing copy.

$$\mathcal{L}_{\text{SFT}}(\theta) = -\mathbb{E}_{(x,y) \sim \mathcal{D}} \left[ \sum_{t=1}^{T} \log \pi_\theta(y_t \mid x, y_{<t}) \right]$$

### SFT Data Preparation

Structure brand-specific marketing examples as conversations. In production, you'd have 500-1000 curated examples per brand.

In [ ]:
# Curated marketing examples as conversations
raw_examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a marketing copywriter for a luxury skincare brand. Tone: sophisticated, confident, never salesy."},
            {"role": "user", "content": "Write an email subject line for our new retinol serum launch."},
            {"role": "assistant", "content": "Your skin has been waiting for this."},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a marketing copywriter for a luxury skincare brand. Tone: sophisticated, confident, never salesy."},
            {"role": "user", "content": "Write a 2-sentence product description for our vitamin C moisturizer."},
            {"role": "assistant", "content": "Radiance isn't something you chase \u2014 it's something you build. Our vitamin C complex works at the cellular level, so the glow you see is the glow you own."},
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a marketing copywriter for a luxury skincare brand. Tone: sophisticated, confident, never salesy."},
            {"role": "user", "content": "Write a social media caption for a before/after skincare transformation."},
            {"role": "assistant", "content": "28 days. Same mirror, different reflection."},
        ]
    },
]

sft_dataset = Dataset.from_list(raw_examples)
print(f"SFT dataset size: {len(sft_dataset)}")
print(f"Example: {sft_dataset[0]}")

### SFT Model & LoRA Setup

We use LoRA to keep fine-tuning feasible on a single GPU. Only a small fraction of parameters are trainable.

In [ ]:
# NOTE: Replace with a smaller model (e.g., "facebook/opt-350m") for testing on CPU
model_name = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

print(f"Model loaded: {model_name}")
print(f"LoRA rank: {lora_config.r}, alpha: {lora_config.lora_alpha}")

### SFT Training

The `SFTTrainer` from TRL handles tokenization, chat template application, and training. Note `max_length=512` — marketing copy is short-form.

In [ ]:
training_args = SFTConfig(
    output_dir="./sft-marketing",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()
trainer.save_model("./sft-marketing-final")
print("SFT training complete!")

## Part 2: RLHF — The Full Pipeline

RLHF uses a three-stage pipeline:
1. SFT (done above)
2. Reward model training on preference pairs
3. RL optimization against the reward model (using RLOO — the modern replacement for PPO)

Reward model loss:
$$\mathcal{L}_{\text{RM}}(\phi) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma\left(r_\phi(x, y_w) - r_\phi(x, y_l)\right) \right]$$

RL objective:
$$\mathcal{L}_{\text{RLHF}}(\theta) = \mathbb{E} \left[ r_\phi(x, y) - \beta \, D_{\text{KL}}\left(\pi_\theta \| \pi_{\text{SFT}}\right) \right]$$

### Preference Data for Reward Model

In marketing, preference pairs come naturally from A/B tests and editorial reviews. The "chosen" copy outperformed or was preferred over the "rejected" version.

In [ ]:
preference_data = [
    {
        "prompt": "Write a flash sale email subject line for 30% off sneakers.",
        "chosen": "Your favorites. 30% off. Today only.",
        "rejected": "Big Sale Alert! Get 30% Off All Sneakers Now!!!",
    },
    {
        "prompt": "Write a push notification for a loyalty reward.",
        "chosen": "You've unlocked something. Check your rewards.",
        "rejected": "Congratulations! You have earned a new loyalty reward! Tap to claim.",
    },
    {
        "prompt": "Write a welcome email subject for a new subscriber.",
        "chosen": "Welcome. Let's make this worth your inbox.",
        "rejected": "Welcome to Our Newsletter! Exciting Updates Await!",
    },
]

reward_dataset = Dataset.from_list(preference_data)
print(f"Reward dataset size: {len(reward_dataset)}")

### Reward Model Training

The reward model learns to assign higher scores to preferred responses. We use a sequence classification head on top of the LLM.

In [ ]:
reward_model_name = "meta-llama/Llama-3.1-8B-Instruct"
reward_tokenizer = AutoTokenizer.from_pretrained(reward_model_name)
reward_tokenizer.pad_token = reward_tokenizer.eos_token

reward_model = AutoModelForSequenceClassification.from_pretrained(
    reward_model_name, num_labels=1, torch_dtype="auto", device_map="auto"
)

print(f"Reward model loaded: {reward_model_name}")

In [ ]:
reward_training_args = RewardConfig(
    output_dir="./reward-model-marketing",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=1e-5,
    max_length=512,
)

reward_trainer = RewardTrainer(
    model=reward_model,
    args=reward_training_args,
    processing_class=reward_tokenizer,
    train_dataset=reward_dataset,
)

reward_trainer.train()
reward_trainer.save_model("./reward-model-marketing-final")
print("Reward model training complete!")

### RLOO Training (RL Stage)

TRL's `RLOOTrainer` (REINFORCE Leave-One-Out) replaces the older PPO approach. It generates multiple completions per prompt, uses the leave-one-out baseline to reduce variance, and is more stable overall. First, prepare the prompt dataset:

In [ ]:
from datasets import Dataset

prompts_data = [
    {"prompt": "Write a subject line for a summer collection launch."},
    {"prompt": "Write a 1-sentence Instagram caption for a new coffee blend."},
    {"prompt": "Write a CTA button label for a free trial signup."},
    {"prompt": "Write a 1-sentence product teaser for wireless earbuds."},
]

prompt_dataset = Dataset.from_list(prompts_data)
print(f"Prompt dataset size: {len(prompt_dataset)}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import RLOOConfig, RLOOTrainer

rl_model = AutoModelForCausalLM.from_pretrained(
    "./sft-marketing-final", torch_dtype="auto", device_map="auto"
)
rl_tokenizer = AutoTokenizer.from_pretrained(model_name)
rl_tokenizer.pad_token = rl_tokenizer.eos_token

rloo_config = RLOOConfig(
    output_dir="./rloo-marketing",
    per_device_train_batch_size=4,
    num_generations=4,           # completions per prompt
    max_completion_length=64,
    learning_rate=1e-6,
    num_train_epochs=1,
    logging_steps=1,
    beta=0.05,                   # KL penalty coefficient
)

rloo_trainer = RLOOTrainer(
    config=rloo_config,
    model=rl_model,
    reward_model=reward_model,
    processing_class=rl_tokenizer,
    reward_processing_class=reward_tokenizer,
    train_dataset=prompt_dataset,
)

rloo_trainer.train()
print("RLOO training complete!")

## Part 3: DPO — Direct Preference Optimization

DPO skips the reward model entirely. It optimizes directly on preference pairs by reparameterizing the reward in terms of the policy:

$$\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right) \right]$$

Same preference data, dramatically simpler pipeline.

### DPO Data

Same format as the reward model data — prompt, chosen, rejected triplets. In marketing, these come from A/B tests and editorial reviews.

In [ ]:
dpo_data = [
    {
        "prompt": "Write a homepage hero headline for an eco-friendly cleaning brand.",
        "chosen": "Clean home. Clean conscience.",
        "rejected": "Our All-Natural Cleaning Products Are Good for You and the Planet!",
    },
    {
        "prompt": "Write a cart abandonment email subject line.",
        "chosen": "Still thinking it over?",
        "rejected": "You Left Items in Your Cart! Complete Your Purchase Now!",
    },
    {
        "prompt": "Write a loyalty program welcome message.",
        "chosen": "You're in. Here's what that means.",
        "rejected": "Welcome to Our Amazing Rewards Program! Start Earning Points Today!",
    },
    {
        "prompt": "Write a seasonal campaign tagline for a winter coat collection.",
        "chosen": "Warmth, engineered.",
        "rejected": "Stay Warm This Winter With Our Cozy Collection of Premium Coats!",
    },
]

dpo_dataset = Dataset.from_list(dpo_data)
print(f"DPO dataset size: {len(dpo_dataset)}")
print(f"Example chosen: {dpo_dataset[0]['chosen']}")
print(f"Example rejected: {dpo_dataset[0]['rejected']}")

### DPO Model Setup

DPO needs the SFT model as both the training model and the reference policy. TRL handles the reference model internally.

In [ ]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

print(f"DPO model loaded: {model_name}")

### DPO Training

One model, one trainer, one loss function. Compare this to the RLHF pipeline above.

In [ ]:
dpo_training_args = DPOConfig(
    output_dir="./dpo-marketing",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    beta=0.1,
    max_length=512,
    logging_steps=10,
)

dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_training_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

dpo_trainer.train()
dpo_trainer.save_model("./dpo-marketing-final")
print("DPO training complete!")

## Summary

| Dimension | SFT | RLHF | DPO |
|-----------|-----|------|-----|
| **Data needed** | Curated examples (500-1k) | Preference pairs (5k+) + SFT data | Preference pairs (1k+) + SFT data |
| **Training complexity** | Single training run | 3-stage pipeline | 2-stage pipeline (SFT then DPO) |
| **GPU memory** | 1 model | 3 models simultaneously | 2 models (policy + reference) |
| **Stability** | Very stable | PPO can diverge | Stable |
| **Captures preferences** | No | Yes | Yes |
| **Time to production** | Days | Weeks | Days |

**For marketing fine-tuning:** Start with SFT to capture brand voice, graduate to DPO for preference learning, reach for RLHF only at scale with rich conversion data.